<a href="https://colab.research.google.com/github/monicavargas3194-coder/Unicafam_Modelos_Analiticos_Indra/blob/main/Sesion_3__Integracion_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from io import StringIO

In [4]:
csv_clientes = """id_cliente;nombre;ciudad;segmento

C001;Ana Ruiz;Bogotá;Oro

C002;Luis Peña;Medellín;Plata

C003;Marta Gil;Cali;Oro

C004;Juan Ríos;Bogotá;Bronce

C005;Sofía Mora;Barranquilla;Plata"""

In [ ]:
df_ =pd.read_csv("")

In [6]:
clientes = pd.read_csv(StringIO(csv_clientes), sep=";")

print(clientes.shape)

print(clientes.head(3))

(5, 4)
  id_cliente     nombre    ciudad segmento
0       C001   Ana Ruiz    Bogotá      Oro
1       C002  Luis Peña  Medellín    Plata
2       C003  Marta Gil      Cali      Oro


In [8]:
respuesta_api = [

{"pedido": 101, "cliente": "C001", "items": 3,

"pago": {"medio": "Tarjeta", "valor": 150000}},

{"pedido": 102, "cliente": "C003", "items": 1,

"pago": {"medio": "PSE", "valor": 80000}},

{"pedido": 103, "cliente": "C001", "items": 2,

"pago": {"medio": "Efectivo", "valor": 45000}},

{"pedido": 104, "cliente": "C009", "items": 5,

"pago": {"medio": "Tarjeta", "valor": 210000}},

]

pedidos = pd.json_normalize(respuesta_api)
print(pedidos)

   pedido cliente  items pago.medio  pago.valor
0     101    C001      3    Tarjeta      150000
1     102    C003      1        PSE       80000
2     103    C001      2   Efectivo       45000
3     104    C009      5    Tarjeta      210000


**merge de datos**

In [11]:
inner = pd.merge(clientes, pedidos, left_on="id_cliente",

right_on="cliente", how="inner")

left = pd.merge(clientes, pedidos, left_on="id_cliente",

right_on="cliente", how="left")




In [12]:

print("INNER:", inner.shape[0], "filas")

print("LEFT :", left.shape[0], "filas")

sin_compra = left[left["pedido"].isna()]["nombre"].tolist()

print("Clientes sin compras:", sin_compra)

INNER: 3 filas
LEFT : 6 filas
Clientes sin compras: ['Luis Peña', 'Juan Ríos', 'Sofía Mora']


In [13]:
outer = pd.merge(clientes, pedidos, left_on="id_cliente",

right_on="cliente", how="outer", indicator=True)

print(outer["_merge"].value_counts())

huerfanos = outer[outer["_merge"] == "right_only"]

print("Pedidos sin cliente:", huerfanos["pedido"].tolist())

_merge
left_only     3
both          3
right_only    1
Name: count, dtype: int64
Pedidos sin cliente: [104.0]


# **consolidacion de datos**


In [15]:
np.random.seed(1)

def ventas_mes(mes):
    return pd.DataFrame({
        "mes": mes,
        "sucursal": ["Norte", "Sur", "Centro"],
        "ventas": np.random.randint(80, 200, 3) * 1e6
    })

consolidado = pd.concat([
    ventas_mes("Ene"),
    ventas_mes("Feb"),
    ventas_mes("Mar")
], ignore_index=True)

print(consolidado.shape)
print(consolidado.tail(3))

(9, 3)
   mes sucursal       ventas
6  Mar    Norte   85000000.0
7  Mar      Sur  159000000.0
8  Mar   Centro  144000000.0


In [16]:
resumen = consolidado.groupby("sucursal").agg(

total=("ventas", "sum"),

promedio=("ventas", "mean"),

meses=("mes", "count"))

resumen = (resumen / [1e6, 1e6, 1]).round(1)

print(resumen.sort_values("total", ascending=False))

          total  promedio  meses
sucursal                        
Sur       435.0     145.0    3.0
Centro    391.0     130.3    3.0
Norte     354.0     118.0    3.0


In [22]:
ancho = consolidado.pivot_table(index="sucursal", columns="mes",
values="ventas") / 1e6

ancho = ancho[["Ene", "Feb", "Mar"]]

print(ancho)



largo = ancho.reset_index().melt(id_vars="sucursal",
var_name="mes", value_name="ventas_M")

print(largo.head(4))

mes         Ene    Feb    Mar
sucursal                     
Centro     92.0  155.0  144.0
Norte     117.0  152.0   85.0
Sur       187.0   89.0  159.0
  sucursal  mes  ventas_M
0   Centro  Ene      92.0
1    Norte  Ene     117.0
2      Sur  Ene     187.0
3   Centro  Feb     155.0


In [23]:
ancho = consolidado.pivot_table(index="sucursal", columns="mes",

values="ventas") / 1e6

ancho = ancho[["Ene", "Feb", "Mar"]]

print(ancho)



largo = ancho.reset_index().melt(id_vars="sucursal",

var_name="mes", value_name="ventas_M")

print(largo.head(4))

mes         Ene    Feb    Mar
sucursal                     
Centro     92.0  155.0  144.0
Norte     117.0  152.0   85.0
Sur       187.0   89.0  159.0
  sucursal  mes  ventas_M
0   Centro  Ene      92.0
1    Norte  Ene     117.0
2      Sur  Ene     187.0
3   Centro  Feb     155.0
